# ImageBind: One Embedding Space To Bind Them All

本教程覆盖 ImageBind 的核心思想与实现，重点演示：

1. **图像作为锚点**：以图像为中心对齐 6 种模态
2. **统一嵌入空间**：所有模态映射到同一向量空间
3. **涌现的跨模态能力**：未直接训练的模态对也能检索
4. **零样本分类与检索**：无需微调即可执行下游任务

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import torch
import torch.nn.functional as F

from imagebind import (
    ImageBindConfig,
    ImageBind,
    ModalityType,
    create_imagebind_model,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. ImageBind 核心理念

ImageBind 的核心思想是**以图像为锚点**，将其他模态对齐到图像空间：

```
        图像 (锚点)
       ╱    │    ╲
      ╱     │     ╲
   文本   音频   深度
     ╲     │     ╱
      ╲    │    ╱
       热力图  IMU
```

**训练数据**：只需要 (图像, X) 配对数据
- 图像-文本：网络图文对
- 图像-音频：视频帧与音轨
- 图像-深度：RGB-D 数据集

**涌现能力**：由于所有模态都与图像对齐，模态之间可以间接对齐
- 文本 ↔ 音频 (通过图像空间传递)

## 2. 模型配置与初始化

In [ ]:
config = ImageBindConfig(
    embed_dim=256,
    vision_embed_dim=384,
    text_embed_dim=256,
    audio_embed_dim=256,
    vision_layers=4,
    vision_heads=6,
    text_layers=4,
    text_heads=4,
    audio_layers=4,
    audio_heads=4,
    imu_layers=2,
    imu_heads=4,
    image_size=112,
    patch_size=14,
    text_vocab_size=1000,
    text_max_length=32,
    audio_num_mel_bins=64,
    audio_target_length=100,
    audio_patch_size=8,
    audio_stride=8,
    imu_seq_length=500,
    imu_patch_size=25,
    modality_embed_dims={
        'image': 384, 'text': 256, 'audio': 256,
        'depth': 384, 'thermal': 384, 'imu': 256,
    },
)

model = ImageBind(config).to(device)
print(f'Model: {model.__class__.__name__}')
print(f'Embed dim: {config.embed_dim}')

## 3. 各模态编码

ImageBind 支持 6 种模态的编码，所有模态输出相同维度的嵌入向量。

In [ ]:
batch_size = 2

# 图像: [B, 3, H, W]
images = torch.randn(batch_size, 3, config.image_size, config.image_size).to(device)
image_embeds = model.encode_image(images)
print(f'Image embeds: {image_embeds.shape}')

# 文本: [B, L]
text_ids = torch.randint(0, config.text_vocab_size, (batch_size, 20)).to(device)
text_embeds = model.encode_text(text_ids)
print(f'Text embeds: {text_embeds.shape}')

# 音频: [B, mel_bins, time]
audio = torch.randn(batch_size, config.audio_num_mel_bins, config.audio_target_length).to(device)
audio_embeds = model.encode_audio(audio)
print(f'Audio embeds: {audio_embeds.shape}')

# 深度图: [B, H, W]
depth = torch.randn(batch_size, config.image_size, config.image_size).to(device)
depth_embeds = model.encode_depth(depth)
print(f'Depth embeds: {depth_embeds.shape}')

# 热力图: [B, 1, H, W]
thermal = torch.randn(batch_size, 1, config.image_size, config.image_size).to(device)
thermal_embeds = model.encode_thermal(thermal)
print(f'Thermal embeds: {thermal_embeds.shape}')

# IMU: [B, T, 6]
imu = torch.randn(batch_size, config.imu_seq_length, 6).to(device)
imu_embeds = model.encode_imu(imu)
print(f'IMU embeds: {imu_embeds.shape}')

## 4. 跨模态相似度计算

所有模态的嵌入都在同一空间，可以直接计算余弦相似度。

In [ ]:
# 图像-文本相似度
sim_image_text = model.compute_similarity(image_embeds, text_embeds)
print(f'Image-Text similarity matrix:\n{sim_image_text}')

# 图像-音频相似度
sim_image_audio = model.compute_similarity(image_embeds, audio_embeds)
print(f'\nImage-Audio similarity matrix:\n{sim_image_audio}')

# 涌现能力：文本-音频相似度 (未直接训练)
sim_text_audio = model.compute_similarity(text_embeds, audio_embeds)
print(f'\nText-Audio similarity (emergent):\n{sim_text_audio}')

## 5. 对比学习训练

ImageBind 使用 InfoNCE 损失进行对比学习。

In [ ]:
# 模拟图像-文本配对训练
batch_images = torch.randn(4, 3, config.image_size, config.image_size).to(device)
batch_texts = torch.randint(0, config.text_vocab_size, (4, 20)).to(device)

outputs = model(
    anchor_modality=ModalityType.IMAGE,
    anchor_input=batch_images,
    positive_modality=ModalityType.TEXT,
    positive_input=batch_texts,
)

print(f'Loss: {outputs["loss"].item():.4f}')
print(f'Anchor embeds shape: {outputs["anchor_embeds"].shape}')
print(f'Positive embeds shape: {outputs["positive_embeds"].shape}')

## 6. 跨模态检索

给定一个模态的查询，检索另一个模态的相关内容。

In [ ]:
# 用图像检索文本
query_image = torch.randn(1, 3, config.image_size, config.image_size).to(device)
gallery_texts = torch.randint(0, config.text_vocab_size, (10, 20)).to(device)

scores, indices = model.retrieve(
    query_modality=ModalityType.IMAGE,
    query_input=query_image,
    gallery_modality=ModalityType.TEXT,
    gallery_inputs=gallery_texts,
    top_k=5,
)

print(f'Top-5 retrieval scores: {scores}')
print(f'Top-5 retrieval indices: {indices}')

## 7. 零样本分类

使用预计算的类别嵌入进行零样本分类。

In [ ]:
# 模拟类别嵌入 (实际应用中由文本编码器生成)
num_classes = 5
class_embeddings = torch.randn(num_classes, config.embed_dim).to(device)
class_embeddings = F.normalize(class_embeddings, dim=-1)

# 对图像进行零样本分类
test_images = torch.randn(3, 3, config.image_size, config.image_size).to(device)
logits = model.zero_shot_classify(
    modality=ModalityType.IMAGE,
    inputs=test_images,
    class_embeddings=class_embeddings,
)

print(f'Logits shape: {logits.shape}')
print(f'Predictions: {logits.argmax(dim=-1)}')
print(f'Probabilities:\n{F.softmax(logits, dim=-1)}')

## 8. 工厂函数

使用预设配置快速创建模型。

In [ ]:
# 创建不同规模的模型
for size in ['tiny', 'small', 'base']:
    m = create_imagebind_model(size)
    num_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'{size}: {num_params:.1f}M parameters, embed_dim={m.config.embed_dim}')

## 9. 关键实现细节

### 9.1 模态编码器
- **ImageEncoder**: ViT 架构，输出 CLS token
- **TextEncoder**: Transformer，输出最后一个有效 token
- **AudioEncoder**: 2D Patch 嵌入处理 Mel 频谱图
- **DepthEncoder/ThermalEncoder**: 与 ImageEncoder 结构相同，单通道输入
- **IMUEncoder**: 1D Patch 嵌入处理时序数据

### 9.2 模态投影器
```python
ModalityProjector:
    Linear(input_dim, output_dim)
    GELU()
    Linear(output_dim, output_dim)
    LayerNorm(output_dim)
```

### 9.3 对比学习损失
```
L = -log(exp(q·k+/τ) / Σexp(q·ki/τ))
```
- 双向对称损失
- 可学习温度参数